# Sesión 1 – Inicio de Chat (Foundry Local)

Este cuaderno inicia Foundry Local, descarga el alias del modelo preferido y realiza tanto una finalización de chat estándar como una finalización de chat en streaming.


# Escenario
Esta sesión introduce lo mínimo necesario para que un modelo de lenguaje pequeño local responda a través de Foundry Local. Harás lo siguiente:
- Instalar las dependencias del SDK / cliente.
- Inicializar el administrador de Foundry Local para un alias elegido (por defecto: `phi-4-mini`).
- Aplicar un monkey‑patch defensivo para tolerar campos opcionales en los metadatos del modelo.
- Enviar una solicitud estándar de finalización de chat.
- Transmitir una respuesta token por token.

El objetivo es validar tu entorno local y la ruta de red antes de pasar a RAG, enrutamiento o agentes.


### Explicación: Instalación de dependencias
Instala los paquetes de Python necesarios para este flujo de chat mínimo:
- `foundry-local-sdk`: Gestiona modelos locales y el ciclo de vida de servicios.
- `openai`: Abstracción de cliente familiar para completar chats.
- `rich`: Impresión bonita para una salida más clara en el notebook.

Volver a ejecutar es seguro (idempotente). Omite este paso si tu entorno ya tiene estos paquetes.


In [1]:
# Install required libraries (idempotent)
%pip install -q foundry-local-sdk openai rich


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Explicación: Importaciones principales
Incluye módulos utilizados en todo el cuaderno:
- `FoundryLocalManager` para interactuar con el entorno de ejecución del modelo local.
- Cliente `OpenAI` para que podamos reutilizar la conocida superficie de la API de finalización de chat.
- `rich.print` para salida con estilo.

No se realizan llamadas a la red aquí; esto solo prepara el espacio de nombres.


In [2]:
import os
from foundry_local import FoundryLocalManager
from foundry_local.models import FoundryModelInfo
from openai import OpenAI
from rich import print

### Explicación: Inicialización del Administrador y Parche de Metadatos
Inicializa `FoundryLocalManager` para el alias elegido y aplica un parche defensivo para manejar de manera adecuada las respuestas del servicio donde `promptTemplate` podría ser `null`.

Resultados clave:
- Confirma el estado del servicio y el endpoint.
- Enumera los modelos almacenados en caché (verifica el almacenamiento local).
- Resuelve el ID concreto del modelo para el alias (utilizado en llamadas de chat posteriores).

Si encuentras problemas de validación en los metadatos del servicio en bruto, este patrón muestra cómo sanitizar sin bifurcar el SDK.


In [3]:
# Monkeypatch to tolerate service responses where promptTemplate is null
_original_from_list_response = FoundryModelInfo.from_list_response

def _safe_from_list_response(response):  # type: ignore
    try:
        if isinstance(response, dict) and response.get("promptTemplate") is None:
            # Normalize to empty dict so pydantic validation passes
            response["promptTemplate"] = {}
    except Exception as e:  # pragma: no cover
        print(f"[yellow]Warning: safe wrapper encountered issue normalizing promptTemplate: {e}[/yellow]")
    return _original_from_list_response(response)

# Apply patch only once
if getattr(FoundryModelInfo.from_list_response, "__name__", "") != "_safe_from_list_response":
    FoundryModelInfo.from_list_response = staticmethod(_safe_from_list_response)  # type: ignore

ALIAS = os.getenv('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini')
manager = FoundryLocalManager(ALIAS)
print(f'[bold green]Service running:[/bold green] {manager.is_service_running()}')
print(f'Endpoint: {manager.endpoint}')
print('Cached models:', manager.list_cached_models())
model_id = manager.get_model_info(ALIAS).id
print(f'Using model id: {model_id}')

Service running: True

Endpoint: http://127.0.0.1:50262/v1

Cached models:
[
    FoundryModelInfo(
        alias='phi-4-mini',
        id='Phi-4-mini-instruct-generic-gpu:4',
        version='4',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-4-mini-instruct-generic-gpu/versions/4',
        file_size_mb=3809,
        prompt_template={
            'system': '<|system|>{Content}<|end|>',
            'user': '<|user|>{Content}<|end|>',
            'assistant': '<|assistant|>{Content}<|end|>',
            'prompt': '<|user|>{Content}<|end|><|assistant|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='qwen2.5-0.5b',
        id='qwen2.5-0.5b-instruct-generic-gpu:3',
        version='3',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/qwen2.5-0.5b-instruct-generic-gpu/versions/3',
        file_size_mb=700,
        prompt_template={
            'system': '<|im_start|>system\n{Content}<|im_end|>',
            'user': '<|im_start|>user\n{Content}<|im_end|>',
            'assistant': '<|im_start|>assistant\n{Content}<|im_end|>',
            'prompt': '<|im_start|>user\n{Content}<|im_end|>\n<|im_start|>assistant'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='apache-2.0',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='phi-3.5-mini',
        id='Phi-3.5-mini-instruct-generic-gpu:1',
        version='1',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-3.5-mini-instruct-generic-gpu/versions/1',
        file_size_mb=2211,
        prompt_template={
            'prompt': '<|user|>\n{Content}<|end|>\n<|assistant|>',
            'assistant': '<|assistant|>\n{Content}<|end|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    )
]

Using model id: Phi-4-mini-instruct-generic-gpu:4

### Explicación: Completar Chat Básico
Crea un cliente compatible con `OpenAI` apuntando al endpoint local de Foundry y realiza una única finalización de chat sin transmisión. Enfoque aquí:
- Asegúrate de que el modelo responda sin errores.
- Valida la latencia / formato de salida.
- Mantén `max_tokens` modesto para conservar recursos.

Si esto falla, verifica nuevamente que el servicio local de Foundry esté en funcionamiento y que el alias se resuelva correctamente.


In [4]:
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key or 'not-needed')
prompt = 'List two benefits of local inference for privacy.'
resp = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':prompt}],
    max_tokens=120,
    temperature=0.5
)
print(resp.choices[0].message.content)

Local inference for privacy refers to the practice of performing data analysis on a local device without sending 
sensitive information to a central server. Two benefits of this approach are:


1. **Enhanced Privacy**: Local inference keeps personal data on the user's device, reducing the risk of data 
breaches and unauthorized access. Since the data is not transmitted over the network, it is less susceptible to 
interception by malicious actors.


2. **Data Sovereignty**: Users retain control over their data, as it does not leave their device. This means that 
individuals or organizations can comply with local data protection regulations, such as the General

### Explicación: Completar Chat en Streaming
Demuestra el streaming de tokens para mejorar la latencia percibida y la experiencia de usuario interactiva. El bucle imprime deltas incrementales a medida que llegan:
- Útil para interfaces de chat donde importa tener una salida parcial temprana.
- Permite medir el rendimiento de tokens frente a la latencia de la finalización completa.

Puedes adaptar este patrón para acumular tokens, actualizar un widget de progreso o abortar la generación a mitad del proceso.


In [5]:
# Streaming example
stream = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':'Give a one-sentence definition of edge AI.'}],
    stream=True,
    max_tokens=60,
    temperature=0.4
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end='', flush=True)
print()

Edge

AI

refers

to

artificial

...

latency

and

bandwidth

usage

.

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por lograr precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o imprecisiones. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables de malentendidos o interpretaciones erróneas que surjan del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
